# Update the course list and schedule

Scrapes every course from kurser.dtu.dk and writes:

| Output | Holds | Used by |
|---|---|---|
| `frontend/data/courses.json` | number and name of every course this year | the survey's course autocomplete |
| `static_data.courses` | the same and each course's department, plus courses DTU no longer offers | `processing/new_proccess_data.ipynb`, for `course` and `department` |
| `frontend/data/courseSchedule.json` | the courses to suggest per weekday, morning and afternoon | the survey's course buttons |

Run it on your own machine when DTU publishes next year's courses, and before each semester for the
schedule. The database is written directly. The frontend is served from the server's checkout, so commit
the two frontend files and `course_abbreviations.json`, then deploy them with `manage.sh update`.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
from sqlalchemy import text

from pis_lib.connect import get_pg_engine
from pis_lib.courses import rank, schedule, scrape
from pis_lib.fetched import record_fetched

SEMESTER = "autumn"  # the semester courseSchedule.json is for: "autumn" or "spring"
MIN_SCORE = 5  # courses scoring less with course_rules.json stay off the schedule
SCRAPED = Path(tempfile.gettempdir()) / "dtu_courses.json"
FRONTEND_DATA = Path("../../frontend/data")

## Scrape

Takes a few minutes and saves the full records to `SCRAPED`, a temporary file, so the cells below can run
again without a new scrape. Interrupting the kernel stops the scrape and saves nothing.

In [ ]:
courses = list(scrape.scrape())
scrape.save(courses, SCRAPED)
print(len(courses), "courses ->", SCRAPED)

## Course list

A course page that fails to load is skipped, and the scrape lists those courses at the end. So does a
course whose student count couldn't be fetched; it is kept, without the enrolment points. The check stops
when the scrape found far fewer courses than the current list.

`static_data.courses` keeps the courses DTU no longer offers, so older answers keep their names.
`static_data.fetched` gets the time of the scrape, for the dashboard.

In [ ]:
courses = json.loads(SCRAPED.read_text())
course_list = [{"number": course["code"], "name": course["name"]} for course in courses]
current = json.loads((FRONTEND_DATA / "courses.json").read_text())
assert len(course_list) >= 0.9 * len(current), f"{len(course_list)} courses, the current list has {len(current)}"

(FRONTEND_DATA / "courses.json").write_text(
    "[\n" + ",\n".join(json.dumps(course, ensure_ascii=False) for course in course_list) + "\n]\n"
)

In [ ]:
new = pd.DataFrame(
    {"course_number": course["code"], "course_name": course["name"], "department": course.get("Department")}
    for course in courses
)
with get_pg_engine().begin() as db:
    db.execute(text("CREATE TABLE IF NOT EXISTS static_data.courses (course_number text, course_name text, department text)"))
    old = pd.read_sql("SELECT * FROM static_data.courses", db)
    pd.concat([new, old]).drop_duplicates("course_number").sort_values("course_number").to_sql(
        "courses", db, schema="static_data", if_exists="replace", index=False
    )
    record_fetched(db, "courses", pd.Timestamp(SCRAPED.stat().st_mtime, unit="s", tz="UTC"))

## Schedule

Each rule in `course_rules.json` gives a course `points` when `pattern`, a case-insensitive regex, matches
its `field` (every text field when `field` is null), or when the number in `field` lies between `min` and
`max`. The bins below show what each course scores; courses scoring at least `MIN_SCORE` with a slot in
`SEMESTER` go on the schedule.

You are asked for a short name for each course the schedule hasn't had before; press enter to keep the
full name. The answers are kept in `course_abbreviations.json`. Two courses with the same name, one taught
in Danish and one in English, get ` (DAN)` and ` (ENG)` added.

In [ ]:
rules = json.loads(Path("course_rules.json").read_text())
rank.print_bins(rank.rank(courses, rules), sys.stdout, head=5)

In [ ]:
timetable = schedule.compile_schedule(courses, rules, "course_abbreviations.json", SEMESTER, MIN_SCORE)
schedule.write_schedule(timetable, FRONTEND_DATA / "courseSchedule.json")